# NB02 – Daten Bereinigung

**Grid-Arbitrage** · Batteriespeicher-Arbitrage im Schweizer Strommarkt

**Gruppe:** SC26_Gruppe_2 | **Verantwortlich:** Senthuran Elankeswaran | **Datum:** März 2026

---

*Bereinigt Spot-Preise: clippt Ausreisser, füllt Lücken, speichert in `data/processed/`.*


| [← NB01 Daten Laden](01_Daten_Laden.ipynb) | [↑ Übersicht ↑](../organisation/O_01_Project_Overview.ipynb) | [NB03 Daten Analyse →](03_Daten_Analyse.ipynb) |
|:---|:---:|---:|

## Inhaltsverzeichnis<a id='toc_NB_02'></a>

1. [Einleitung](#einleitung_NB_02)
2. [Initialisierung](#initialisierung_NB_02)
3. [Datenbereinigung Spot-Preise](#datenbereinigung-spot-preise_NB_02)
4. [Fazit](#fazit_NB_02)
5. [Abschluss](#abschluss_NB_02)


---
## Einleitung <a id='einleitung_NB_02'></a>

[↑ Inhaltsverzeichnis](#toc_NB_02)

Bereinigung der ENTSO-E-Rohdaten aus NB01:

- **UTC-Normierung** — alle Zeitstempel auf UTC setzen (vermeidet Probleme durch
  Sommer-/Winterzeit-Umstellung und doppelte/fehlende Stunden)
- **Deduplizierung** — doppelte Stunden bei der Zeitumstellung Oktober entfernen
- **Kurzlücken schliessen** — fehlende Einzelwerte (<3h) interpolieren,
  längere Lücken bleiben als `NaN` für Nachvollziehbarkeit

Output: `data/processed/spot_prices_clean.csv` — Grundlage für Dispatch- und
Wirtschaftlichkeitsrechnung in NB03.


## Initialisierung<a id='initialisierung_NB_02'></a>

[↑ Inhaltsverzeichnis](#toc_NB_02)

Lädt `../sync/config.json` (SSOT), setzt Verzeichnispfade, liest Simulation- und Wirtschaftlichkeitsparameter sowie Datenzeitraum aus `../sync/transfer.json`.

**Setup – Konfiguration & Verzeichnisstruktur:** Lädt `../sync/config.json`, setzt Pfade
und liest `FORCE_RELOAD`, `LIFETIME` und `ZIEL_ROI` als lokale Aliases.


In [ ]:
# ── lib/ aus Projekt-Root erreichbar machen + lib-Imports ───────────────────
# Notebook liegt in einem Unterordner (kuer/, experimental/, notebooks/,
# organisation/). Damit 'from lib.xxx import ...' funktioniert, muss der
# Projekt-Root vorne in sys.path stehen. autoreload sorgt dafür, dass
# Änderungen in lib/*.py ohne Kernel-Restart übernommen werden.
import sys, os
_PROJECT_ROOT = os.path.abspath('..')
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

# lib-Imports (einmal zentral — in allen folgenden Zellen verfügbar)
from lib.plotting import show_source
from lib.io_ops   import load_transfer, save_transfer, log_dataindex, needs_rebuild, log_missing

print(f'lib-Pfad aktiv: {_PROJECT_ROOT}/lib')


In [ ]:
import os, sys, warnings, json
import numpy  as np
import pandas as pd
from datetime import datetime
warnings.filterwarnings('ignore')

# Versionen anzeigen für Reproduzierbarkeit
print(f"Numpy        Version: {np.__version__}")
print(f"Pandas       Version: {pd.__version__}")
print(f"📅 Zuletzt ausgeführt am: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")

**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `load_transfer` wird aus `lib/io_ops.py` importiert und
liest Einträge aus `sync/transfer.json`. Aufklappbar ist der Quellcode einsehbar.


In [ ]:
show_source(load_transfer)


In [ ]:
# ── ../sync/config.json laden (Single Source of Truth) ───────────────────────────────
# NEVER hardcode MODE oder FORCE_RELOAD hier — immer aus ../sync/config.json lesen
with open('../sync/config.json') as _f:
    CFG = json.load(_f)

MODE         = CFG['mode']
FORCE_RELOAD = CFG['force_reload']

# ── Sim-Parameter aus ../sync/config.json ────────────────────────────────────────────
_sim         = CFG['pflicht']['simulation']
CHARGE_Q     = _sim['charge_quantile']      # 0.25
DISCHARGE_Q  = _sim['discharge_quantile']   # 0.75
SOC_MIN_PCT  = _sim['soc_min_pct']          # 0.05
SOC_MAX_PCT  = _sim['soc_max_pct']          # 0.95
EFFICIENCY   = _sim['efficiency_roundtrip'] # 0.92

# ── Wirtschaftlichkeits-Parameter ─────────────────────────────────────────────
_w           = CFG['pflicht']['wirtschaftlichkeit']
CAPEX_EUR_KWH = _w['capex_eur_kwh']
OPEX_RATE    = _w['opex_rate']
LIFETIME     = _w['lifetime_j']
ZIEL_ROI_PCT = round(100 / LIFETIME, 2)  # Ziel-ROI = 1/lifetime_j: ROI der nötig ist, um BE in LIFETIME Jahren zu erreichen

# ── Szenario-Parameter (Gleichzeitigkeit) ─────────────────────────────────────
_sz           = CFG['szenarien']
GLEICHZEITIGKEIT = _sz['gleichzeitigkeit_aktiv']          # 'pessimistisch'|'realistisch'|'optimistisch'
RATE          = _sz['optionen'][GLEICHZEITIGKEIT]['rate']
SZ_OPT        = _sz['optionen'][GLEICHZEITIGKEIT]         # alle Parameter des aktiven Szenarios
CH_SPITZENLAST_GW = _sz['ch_spitzenlast_gw']


In [ ]:
DIR_RAW          = os.path.join('../data', 'raw')
DIR_PROCESSED    = os.path.join('../data', 'processed')
# intermediate: Szenario-abhängige Dateien in Unterordner
DIR_INTER_BASE   = os.path.join('../data', 'intermediate')
DIR_INTER_SZ     = os.path.join(DIR_INTER_BASE, GLEICHZEITIGKEIT)   # z.B. data/intermediate/realistisch/
DIR_INTER        = DIR_INTER_BASE  # szenario-unabhängige Dateien (spread, import/export)
DATAINDEX        = '../sync/dataindex.csv'

for d in [DIR_RAW, DIR_PROCESSED, DIR_INTER_BASE, DIR_INTER_SZ]:
    os.makedirs(d, exist_ok=True)

print(f'../sync/config.json  : geladen')
print(f'MODE         : {MODE}')
print(f'Gleichz.     : {GLEICHZEITIGKEIT} ({RATE*100:.0f}%)')
print(f'raw          : {os.path.abspath(DIR_RAW)}')
print(f'processed    : {os.path.abspath(DIR_PROCESSED)}')
print(f'intermediate : {os.path.abspath(DIR_INTER_SZ)} (szenario-abhängig)')


In [ ]:
# -- Transfer: Ergebnisse aus ../sync/transfer.json laden ----------------------------
TF       = load_transfer()
_dt      = TF.get('datenzeitraum', {})
_sim_tf  = TF.get('simulation', {})
TF_N_YEARS = _dt.get('n_years', None)
TF_START   = _dt.get('start_date', 'unbekannt')
TF_END     = _dt.get('end_date',   'unbekannt')
TF_SPREAD  = _sim_tf.get('spread_mean_eur_mwh', None)
TF_ECON    = _sim_tf.get('wirtschaftlichkeit', {})
TF_HYB     = TF.get('hybrid_simulation', {}).get('ergebnisse', {})
TF_KUER    = CFG.get('kuer_aktiv', {})
if TF:
    print(f"../sync/transfer.json: {TF_START} – {TF_END} ({TF_N_YEARS} Jahre) | Spread: {TF_SPREAD} EUR/MWh")


**Transfer NB01 → NB02:** Datenzeitraum (`n_years`) aus `../sync/transfer.json` lesen —
SSOT für alle Jahresdurchschnitts-Berechnungen. Fehlt die Datei: Fallback auf Selbstberechnung.


In [ ]:
# -- Transfer: n_years von NB01 übernehmen (SSOT: gleiche Methode überall) ----
_dt_r  = load_transfer(key='datenzeitraum', default={})
n_years = _dt_r.get('n_years', None)
_sz_tf  = _dt_r.get('_szenario_aktiv', None)
if n_years:
    print(f'n_years aus ../sync/transfer.json (NB01): {n_years:.3f}')
if _sz_tf and _sz_tf != SZ_AKTIV:
    print(f'⚠️  Szenario-Konflikt: transfer={_sz_tf} vs config={SZ_AKTIV}')
    print('   NB01 mit aktuellem Szenario neu ausführen!')


**Hilfsfunktionen:** `log_dataindex()` für das Datenprotokoll (identisch zu NB01,
hier neu definiert damit NB02 eigenständig ausführbar bleibt).


**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `log_dataindex` wird aus `lib/io_ops.py` importiert und
schreibt Einträge ins Daten-Provenienz-Protokoll `sync/dataindex.csv`.
Bereits bestehende Einträge für denselben Dateinamen werden als
`superseded` markiert. Aufklappbar darunter ist der Quellcode einsehbar.


In [ ]:
show_source(log_dataindex)


In [ ]:
# ── ../sync/dataindex.csv Helper ───────────────────────────────────────────────────────
print('dataindex Helper bereit.')

print('Helpers bereit.')


**Rohdaten laden:** Spot-Preise und Netzlast aus `data/raw/` einlesen;
Zeitfeatures (`hour`, `month`, `season`) berechnen.


In [ ]:
# ── Rohdaten laden ────────────────────────────────────────────────────────────
df_prices = pd.read_csv(os.path.join(DIR_RAW, 'ch_spot_prices_raw.csv'))
df_load   = pd.read_csv(os.path.join(DIR_RAW, 'ch_netzlast_raw.csv'))
print(f'MODE       : {MODE}')
print(f'Preisdaten : {len(df_prices):,} Zeilen | {list(df_prices.columns)}')
print(f'Lastdaten  : {len(df_load):,} Zeilen  | {list(df_load.columns)}')


---
## 1. Datenbereinigung Spot-Preise <a id='datenbereinigung-spot-preise_NB_02'></a>

[↑ Inhaltsverzeichnis](#toc_NB_02)

Clippt physikalisch unplausible Preise (< −500 / > 3 000 EUR/MWh), füllt
Lücken per Forward-Fill und ergänzt Zeitfeatures (`hour`, `month`, `season`).
Ergebnis: `ch_spot_prices_clean.csv` in `processed/`.


**🔎 Quellcode der importierten lib-Funktion**

`needs_rebuild` aus `lib.io_ops` — aufklappbar ist der Quellcode einsehbar.


In [ ]:
show_source(needs_rebuild)


In [ ]:
# FORCE_RELOAD['clean'] = True → Zelle neu ausführen erzwingt Neuberechnung
CLEAN_FILE = os.path.join(DIR_PROCESSED, 'ch_spot_prices_clean.csv')

if needs_rebuild(CLEAN_FILE, 17000, 'clean'):
    # df_prices: Rohdaten aus vorheriger Zelle
    df_prices = pd.read_csv(os.path.join(DIR_RAW, 'ch_spot_prices_raw.csv'),
                             parse_dates=['timestamp'])
    # 1a: UTC normieren
    df_prices['timestamp'] = pd.to_datetime(df_prices['timestamp'], utc=True)
    df_prices = df_prices.sort_values('timestamp').reset_index(drop=True)
    # 1b: Vollständigen Stundenindex erzwingen
    full_idx = pd.date_range(df_prices['timestamp'].min(),
                              df_prices['timestamp'].max(), freq='1h', tz='UTC')
    df_prices = df_prices.set_index('timestamp').reindex(full_idx).reset_index()
    df_prices.rename(columns={'index': 'timestamp'}, inplace=True)
    print(f'1b – Fehlende Stunden: {df_prices["price_eur_mwh"].isna().sum()}')
    # 1c–1e: Interpolieren, Ausreisser, Zeitfeatures
    df_prices['price_eur_mwh'] = (df_prices['price_eur_mwh']
        .interpolate(method='linear', limit=3)
        .fillna(method='ffill', limit=6).fillna(method='bfill', limit=6))
    n_clip = ((df_prices['price_eur_mwh'] < -500) | (df_prices['price_eur_mwh'] > 3000)).sum()
    df_prices['price_eur_mwh'] = df_prices['price_eur_mwh'].clip(-500, 3000)
    print(f'1d – Ausreisser: {n_clip}')
    df_prices['hour']    = df_prices['timestamp'].dt.hour
    df_prices['month']   = df_prices['timestamp'].dt.month
    df_prices['weekday'] = df_prices['timestamp'].dt.dayofweek
    df_prices['season']  = df_prices['month'].map(
        {12:0,1:0,2:0,3:1,4:1,5:1,6:2,7:2,8:2,9:3,10:3,11:3})
    # 1f: Speichern
    df_prices.to_csv(CLEAN_FILE, index=False)
    kb = os.path.getsize(CLEAN_FILE) / 1024
    log_dataindex('ch_spot_prices_clean.csv', 'NB2:bereinigung', CLEAN_FILE, 'processed',
                  rows=len(df_prices), size_kb=kb, note=f'MODE={MODE}')
    print(f'Bereinigt → {CLEAN_FILE} | {len(df_prices):,} h | {kb:.0f} KB')
else:
    df_prices = pd.read_csv(CLEAN_FILE, parse_dates=['timestamp'])
    print(f'ch_spot_prices_clean vorhanden ({len(df_prices):,} Z) – Bereinigung übersprungen.\nFür ein erneutes Rebuild config.json force_reload.clean auf true setzen')

**Verifikation:** Shape, Nullwerte und Wertebereich der bereinigten Preise prüfen.


In [ ]:
# ── Verifikation: Bereinigte Preise ─────────────────────────────────────────
print(f'Shape   : {df_prices.shape}')
print(f'Dtypes  : timestamp={df_prices["timestamp"].dtype}, '
      f'price={df_prices["price_eur_mwh"].dtype}')
print(f'Features: {[c for c in df_prices.columns if c not in ["timestamp","price_eur_mwh"]]}')
print(f'Nulls   : {df_prices.isnull().sum().sum()}')
df_prices.head(3)


---
## Fazit <a id='fazit_NB_02'></a>

[↑ Inhaltsverzeichnis](#toc_NB_02)

Bereinigte Preisreihe ohne Duplikate und mit gefüllten Kurzlücken vorhanden.
Datenqualität: >99% Abdeckung pro Jahr, verbleibende `NaN` sind dokumentiert.
Die Reihe ist der Input für Dispatch-Simulation (NB03) und alle Kür-Analysen.


---
## Abschluss <a id='abschluss_NB_02'></a>

[↑ Inhaltsverzeichnis](#toc_NB_02)

Pflicht-Ausgabedateien auf Existenz und Mindestgrösse prüfen.
Fehler müssen vor NB03 behoben werden.


In [ ]:
# ── Abschlusskontrolle NB02 ──────────────────────────────────────────────────
print('NB02 – Abschlusskontrolle')
print('=' * 60)
CLEAN_FILE = os.path.join(DIR_PROCESSED, 'ch_spot_prices_clean.csv')
all_ok = True
for fpath, min_kb, desc in [
    (CLEAN_FILE, 100, 'Bereinigte Spot-Preise (ch_spot_prices_clean.csv)'),
]:
    ok = os.path.exists(fpath) and os.path.getsize(fpath)/1024 >= min_kb
    icon = '✅' if ok else '❌'
    print(f'  {icon} {desc}')
    if not ok: all_ok = False
print()
print('→ Weiter mit NB03 Daten Analyse.' if all_ok else '→ Fehler beheben vor NB03.')


| [← NB01 Daten Laden](01_Daten_Laden.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [NB03 Daten Analyse →](03_Daten_Analyse.ipynb) |
|:---|:---:|---:|